# Preventing Data Leakage in Simulated TCSPC Data

This notebook investigates how train-test splitting strategies affect the evaluation of machine-learning models trained on simulated TCSPC decay curves.

Synthetic datasets often contain several Poisson-noisy realizations generated from the same underlying physical parameters. If these closely related curves are randomly distributed between the training and test sets, the resulting evaluation may be unrealistically optimistic.

The notebook compares:

* an ordinary random row-level split;
* a group-aware split based on simulation parameter sets;
* a lifetime-interval holdout experiment;
* a noise-robustness experiment.

The purpose is to distinguish performance on familiar parameter regions from robustness to genuinely unfamiliar physical or statistical conditions.

## 1. Scientific motivation

A supervised machine-learning model is normally trained on one subset of a dataset and evaluated on another subset that was not used during training.

For the evaluation to be scientifically meaningful, the test samples should represent genuinely unseen information.

In simulated TCSPC datasets, however, multiple measured curves may be generated from the same expected decay curve. For example, several Poisson-noisy realizations may share exactly the same:

* fluorescence lifetime;
* decay amplitude;
* background level;
* time axis;
* decay model.

Only the random Poisson realization differs between these curves.

Consider three simulated curves generated using the same physical parameters:

- lifetime: 2.5 ns
- amplitude: 10,000 counts
- background: 5 counts per bin

The corresponding measured curves may differ because independent Poisson noise is applied:

| Curve | Parameter set | Random seed |
|---:|---|---:|
| 1 | A | 101 |
| 2 | A | 102 |
| 3 | A | 103 |

An ordinary random train-test split may assign curves 1 and 2 to the training set and curve 3 to the test set.

Although curve 3 was not directly used during training, it originates from the same expected decay curve as curves 1 and 2.

The model is therefore tested on a sample that is statistically new but physically almost identical to samples already present in the training set.

This problem is not classical target leakage because the true lifetime is not directly included among the model input features.

Instead, it is a form of **simulation-group leakage** or **dependence between training and test samples**.

The training and test sets are not sufficiently independent because they contain different noisy realizations of the same underlying simulation parameter set.

A model evaluated under these conditions may achieve a very small test error without demonstrating that it can generalize to:

* unseen lifetime values;
* unseen combinations of lifetime, amplitude, and background;
* lower photon-count conditions;
* stronger background noise;
* different decay models;
* unfamiliar instrument-response functions.

It is therefore important to define what kind of generalization each evaluation experiment is intended to measure.

In this notebook, four generalization concepts are distinguished.

### Interpolation performance

The test data lie within the physical parameter ranges represented during training, but the exact simulation parameter combinations are new.

### Robustness to new noise levels

The physical decay model remains unchanged, but the test curves contain photon counts or background levels not represented during training.

### Robustness to model mismatch

The test curves do not exactly follow the decay model used to generate the training data. For example, a model trained on mono-exponential curves may later be tested on bi-exponential curves.

### Robustness to unfamiliar IRFs

After instrument-response-function convolution is added to the toolkit, a model may be trained using one family of IRFs and evaluated using different IRF
shapes or widths.

The main question throughout this notebook is:

> What exactly is unfamiliar to the model in the test set?

A test set may differ from the training set only through a new Poisson realization, or it may introduce a genuinely new physical or statistical
condition.

These situations support different scientific claims and should therefore be evaluated separately.

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.ensemble import (
    GradientBoostingRegressor,
    RandomForestRegressor,
)
from sklearn.linear_model import Ridge
from sklearn.metrics import (
    mean_absolute_error,
    median_absolute_error,
    r2_score,
)
from sklearn.model_selection import (
    GroupShuffleSplit,
    train_test_split,
)